<div style="display: flex; align-items: center;">

  <!-- Logos -->
  <div style="white-space: nowrap;">
    <img 
      src="https://www.upc.edu/comunicacio/ca/identitat/descarrega-arxius-grafics/fitxers-marca-principal/upc-positiu-p3005.png" 
      width="300"
      style="vertical-align: middle;"
    >
    <img 
      src="https://www.hipotecalowcost.com/wp-content/uploads/2019/08/Logo-CaixaBank.png" 
      width="200"
      style="vertical-align: middle;"
    >
  </div>

  <!-- Texto -->
  <div style="margin-left: auto; margin-right: 10px; text-align: right;">
      <p style="margin: 0;"><b>CaixaBank · Advanced Analytics Program</b></p>
      <p style="margin: 0;"><b>Model Risk & Data Science Training</b></p>
      <p style="margin: 0;">Automatización de procesos</p>
  </div>

</div>

# **Funciones, parámetros, retornos, modularización y reutilización**
## *Del script monolítico al código mantenible*

En este laboratorio transformaremos automatizaciones funcionales pero difíciles de mantener en código reutilizable.

Seguiremos la secuencia:

> **script monolítico → detectar problemas → extraer funciones → parametrizar → devolver resultados → validar → modularizar → reutilizar**

El objetivo no es simplemente “usar funciones”, sino diseñarlas con responsabilidades claras.

## Objetivos

Al finalizar deberías poder:

1. Explicar por qué una función mejora mantenibilidad.
2. Diseñar funciones con una única responsabilidad.
3. Utilizar parámetros posicionales y con nombre.
4. Utilizar valores por defecto.
5. Diferenciar `return` y efectos secundarios.
6. Devolver múltiples resultados.
7. Evitar variables globales innecesarias.
8. Validar argumentos.
9. Reutilizar funciones en distintos flujos.
10. Separar lógica y orquestación.
11. Crear módulos `.py`.
12. Importar funciones propias.
13. Diseñar una pequeña API interna para automatización documental.

## 1. Preparación

In [ ]:
from pathlib import Path
import importlib.util
import shutil
import sys
import textwrap

import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 50)

print("Entorno preparado.")

# 2. Partimos de un script monolítico

In [ ]:
archivos = [
    "factura_001.pdf",
    "clientes.xlsx",
    "foto.jpg",
    "datos.csv",
    "README",
    "backup.zip",
    "factura_002.PDF",
    "notas.txt"
]

reglas = {
    ".pdf": "PDF",
    ".xlsx": "Excel",
    ".csv": "Datos",
    ".jpg": "Imagenes",
    ".txt": "Texto"
}

registros = []

for archivo in archivos:
    extension = Path(
        archivo
    ).suffix.lower()

    if not extension:
        categoria = "Sin_extension"
    else:
        categoria = reglas.get(
            extension,
            "Revision"
        )

    estado = (
        "revision"
        if categoria
        in {
            "Revision",
            "Sin_extension"
        }
        else "automatico"
    )

    registros.append({
        "archivo": archivo,
        "extension": (
            extension
            if extension
            else "sin_extension"
        ),
        "categoria": categoria,
        "estado": estado
    })

resultado_monolitico = pd.DataFrame(
    registros
)

resultado_monolitico

## 2.1 ¿Qué problemas tiene?

El código funciona, pero concentra demasiadas responsabilidades:

- recorrer;
- extraer extensión;
- clasificar;
- decidir estado;
- construir registro;
- acumular resultados.

Esto dificulta cambiar una regla sin afectar otras partes.

# 3. Primera extracción: obtener extensión

In [ ]:
def obtener_extension(
    nombre_archivo
):
    extension = (
        Path(
            nombre_archivo
        )
        .suffix
        .lower()
    )

    if not extension:
        return "sin_extension"

    return extension


print(
    obtener_extension(
        "FACTURA.PDF"
    )
)

print(
    obtener_extension(
        "README"
    )
)

### Interpretación

La función:

- recibe una entrada;
- aplica una regla concreta;
- devuelve un resultado.

No mueve archivos ni imprime ni modifica estructuras externas.

# 4. Parámetros y retorno

In [ ]:
def clasificar_extension(
    extension,
    reglas_clasificacion
):
    if extension == "sin_extension":
        return "Sin_extension"

    return reglas_clasificacion.get(
        extension,
        "Revision"
    )


print(
    clasificar_extension(
        ".pdf",
        reglas
    )
)

print(
    clasificar_extension(
        ".zip",
        reglas
    )
)

# 5. Decidir estado

In [ ]:
def decidir_estado(
    categoria
):
    if categoria in {
        "Revision",
        "Sin_extension"
    }:
        return "revision"

    return "automatico"


print(
    decidir_estado(
        "PDF"
    )
)

print(
    decidir_estado(
        "Revision"
    )
)

# 6. Componer funciones

In [ ]:
def analizar_archivo(
    archivo,
    reglas_clasificacion
):
    extension = obtener_extension(
        archivo
    )

    categoria = clasificar_extension(
        extension,
        reglas_clasificacion
    )

    estado = decidir_estado(
        categoria
    )

    return {
        "archivo": archivo,
        "extension": extension,
        "categoria": categoria,
        "estado": estado
    }


analizar_archivo(
    "backup.zip",
    reglas
)

### Interpretación

`analizar_archivo()` orquesta funciones más pequeñas.

Cada una tiene una responsabilidad concreta.

# 7. Procesar una colección

In [ ]:
def procesar_archivos(
    archivos_entrada,
    reglas_clasificacion
):
    return [
        analizar_archivo(
            archivo,
            reglas_clasificacion
        )
        for archivo
        in archivos_entrada
    ]


resultado_funcional = pd.DataFrame(
    procesar_archivos(
        archivos,
        reglas
    )
)

resultado_funcional

# 8. Validar equivalencia con el script inicial

In [ ]:
columnas = [
    "archivo",
    "extension",
    "categoria",
    "estado"
]

pd.testing.assert_frame_equal(
    resultado_monolitico[
        columnas
    ].reset_index(
        drop=True
    ),
    resultado_funcional[
        columnas
    ].reset_index(
        drop=True
    )
)

print(
    "Mismo resultado."
)

### Regla fundamental

Refactorizar no debería cambiar el comportamiento si no queremos cambiar funcionalidad.

# 9. Parámetros por defecto

In [ ]:
def clasificar_extension(
    extension,
    reglas_clasificacion,
    categoria_desconocida="Revision"
):
    if extension == "sin_extension":
        return "Sin_extension"

    return reglas_clasificacion.get(
        extension,
        categoria_desconocida
    )


print(
    clasificar_extension(
        ".zip",
        reglas
    )
)

print(
    clasificar_extension(
        ".zip",
        reglas,
        categoria_desconocida="Comprimidos"
    )
)

Un valor por defecto permite que la función tenga un comportamiento estándar, pero siga siendo configurable.

# 10. Argumentos con nombre

In [ ]:
resultado = clasificar_extension(
    extension=".zip",
    reglas_clasificacion=reglas,
    categoria_desconocida="Pendiente"
)

resultado

Los argumentos con nombre mejoran legibilidad cuando una función tiene varios parámetros con significados distintos.

# 11. Provocar un argumento inválido

In [ ]:
try:
    obtener_extension(
        None
    )

except Exception as error:
    print(
        type(error).__name__,
        "->",
        error
    )

### Problema

El error aparece dentro de `Path`, lejos del significado de nuestra función.

Podemos validar antes y devolver un mensaje más útil.

# 12. Validar argumentos

In [ ]:
def obtener_extension_segura(
    nombre_archivo
):
    if not isinstance(
        nombre_archivo,
        str
    ):
        raise TypeError(
            "nombre_archivo debe ser str."
        )

    if not nombre_archivo.strip():
        raise ValueError(
            "nombre_archivo no puede estar vacío."
        )

    extension = (
        Path(
            nombre_archivo
        )
        .suffix
        .lower()
    )

    return (
        extension
        if extension
        else "sin_extension"
    )


for valor in [
    None,
    "",
    "factura.pdf"
]:
    try:
        print(
            valor,
            "->",
            obtener_extension_segura(
                valor
            )
        )

    except Exception as error:
        print(
            valor,
            "->",
            type(error).__name__,
            error
        )

# 13. `return` frente a `print`

In [ ]:
def calcular_total_mal(
    unidades,
    precio
):
    print(
        unidades
        * precio
    )


resultado = calcular_total_mal(
    3,
    10
)

print(
    "Valor devuelto:",
    resultado
)

### Diagnóstico

La función muestra el valor pero devuelve `None`.

Esto dificulta reutilizar el resultado en un pipeline.

# 14. Corrección con `return`

In [ ]:
def calcular_total(
    unidades,
    precio
):
    return (
        unidades
        * precio
    )


resultado = calcular_total(
    3,
    10
)

print(
    resultado
)

print(
    resultado * 2
)

# 15. Devolver múltiples resultados

In [ ]:
def resumir_lote(
    tabla
):
    total = len(
        tabla
    )

    automaticos = int(
        (
            tabla[
                "estado"
            ]
            == "automatico"
        ).sum()
    )

    revision = int(
        (
            tabla[
                "estado"
            ]
            == "revision"
        ).sum()
    )

    return (
        total,
        automaticos,
        revision
    )


total, automaticos, revision = (
    resumir_lote(
        resultado_funcional
    )
)

print(
    total,
    automaticos,
    revision
)

# 16. Mejor alternativa: devolver un diccionario

In [ ]:
def calcular_kpis(
    tabla
):
    total = len(
        tabla
    )

    automaticos = int(
        (
            tabla[
                "estado"
            ]
            == "automatico"
        ).sum()
    )

    revision = int(
        (
            tabla[
                "estado"
            ]
            == "revision"
        ).sum()
    )

    return {
        "total":
            total,
        "automaticos":
            automaticos,
        "revision":
            revision,
        "tasa_automatizacion":
            (
                automaticos
                / total
                if total
                else 0
            )
    }


kpis = calcular_kpis(
    resultado_funcional
)

kpis

### Interpretación

Un diccionario devuelve resultados con nombre, lo que reduce errores de posición y mejora legibilidad.

# 17. Efectos secundarios

In [ ]:
procesados_globales = []


def registrar_mal(
    archivo
):
    procesados_globales.append(
        archivo
    )


registrar_mal(
    "factura.pdf"
)

procesados_globales

### Problema

La función modifica una variable externa.

Eso introduce dependencia oculta y dificulta pruebas y reutilización.

# 18. Alternativa explícita

In [ ]:
def registrar(
    registros_actuales,
    archivo
):
    nuevos = (
        registros_actuales.copy()
    )

    nuevos.append(
        archivo
    )

    return nuevos


procesados = []

procesados = registrar(
    procesados,
    "factura.pdf"
)

procesados

No siempre debemos copiar estructuras grandes, pero el ejemplo muestra una idea importante:

> los efectos secundarios deben ser conscientes y explícitos.

# 19. Funciones pequeñas vs función gigante

In [ ]:
def procesar_todo_mal(
    archivos_entrada,
    reglas_clasificacion
):
    salida = []

    for archivo in archivos_entrada:
        extension = (
            Path(
                archivo
            )
            .suffix
            .lower()
        )

        if not extension:
            extension = (
                "sin_extension"
            )

        if extension == "sin_extension":
            categoria = (
                "Sin_extension"
            )

        else:
            categoria = (
                reglas_clasificacion
                .get(
                    extension,
                    "Revision"
                )
            )

        if categoria in {
            "Revision",
            "Sin_extension"
        }:
            estado = "revision"
        else:
            estado = "automatico"

        salida.append({
            "archivo":
                archivo,
            "extension":
                extension,
            "categoria":
                categoria,
            "estado":
                estado
        })

    return pd.DataFrame(
        salida
    )


procesar_todo_mal(
    archivos,
    reglas
)

Esta función no es necesariamente incorrecta, pero concentra demasiadas decisiones.

La versión compuesta es más fácil de:

- probar;
- cambiar;
- documentar;
- reutilizar.

# 20. Reutilizar la misma lógica con reglas distintas

In [ ]:
reglas_finanzas = {
    ".pdf": "Documentos",
    ".xlsx": "Finanzas",
    ".csv": "Finanzas"
}

reglas_marketing = {
    ".jpg": "Creatividades",
    ".png": "Creatividades",
    ".pdf": "Material"
}

ejemplo = [
    "informe.pdf",
    "datos.csv",
    "banner.png"
]

print(
    pd.DataFrame(
        procesar_archivos(
            ejemplo,
            reglas_finanzas
        )
    )
)

print()

print(
    pd.DataFrame(
        procesar_archivos(
            ejemplo,
            reglas_marketing
        )
    )
)

### Interpretación

La lógica no cambia.

Cambian los datos de configuración.

Esto es una señal de buen diseño.

# 21. Separar orquestación de lógica

In [ ]:
def ejecutar_pipeline_documental(
    archivos_entrada,
    reglas_clasificacion
):
    registros = procesar_archivos(
        archivos_entrada,
        reglas_clasificacion
    )

    tabla = pd.DataFrame(
        registros
    )

    metricas = calcular_kpis(
        tabla
    )

    return {
        "tabla":
            tabla,
        "metricas":
            metricas
    }


salida = ejecutar_pipeline_documental(
    archivos,
    reglas
)

salida[
    "tabla"
]

In [ ]:
salida[
    "metricas"
]

# 22. Visualizar métricas

In [ ]:
metricas = salida[
    "metricas"
]

plt.figure(
    figsize=(7, 5)
)

plt.bar(
    [
        "Automáticos",
        "Revisión"
    ],
    [
        metricas[
            "automaticos"
        ],
        metricas[
            "revision"
        ]
    ]
)

plt.ylabel(
    "Archivos"
)

plt.title(
    "Resultado del pipeline"
)

plt.tight_layout()
plt.show()

# 23. Convertir funciones en módulo

In [ ]:
proyecto = Path(
    "demo_modulos"
)

if proyecto.exists():
    shutil.rmtree(
        proyecto
    )

proyecto.mkdir()

ruta_modulo = (
    proyecto
    / "documentos.py"
)

codigo_modulo = '''
from pathlib import Path

def obtener_extension(nombre_archivo):
    extension = Path(nombre_archivo).suffix.lower()

    return extension if extension else "sin_extension"


def clasificar_extension(
    extension,
    reglas_clasificacion,
    categoria_desconocida="Revision"
):
    if extension == "sin_extension":
        return "Sin_extension"

    return reglas_clasificacion.get(
        extension,
        categoria_desconocida
    )


def decidir_estado(categoria):
    if categoria in {
        "Revision",
        "Sin_extension"
    }:
        return "revision"

    return "automatico"


def analizar_archivo(
    archivo,
    reglas_clasificacion
):
    extension = obtener_extension(
        archivo
    )

    categoria = clasificar_extension(
        extension,
        reglas_clasificacion
    )

    return {
        "archivo": archivo,
        "extension": extension,
        "categoria": categoria,
        "estado": decidir_estado(
            categoria
        )
    }
'''

ruta_modulo.write_text(
    textwrap.dedent(
        codigo_modulo
    ),
    encoding="utf-8"
)

print(
    ruta_modulo.read_text(
        encoding="utf-8"
    )
)

# 24. Importar nuestro módulo

In [ ]:
spec = (
    importlib.util
    .spec_from_file_location(
        "documentos",
        ruta_modulo
    )
)

documentos_modulo = (
    importlib.util
    .module_from_spec(
        spec
    )
)

spec.loader.exec_module(
    documentos_modulo
)

documentos_modulo.analizar_archivo(
    "factura.PDF",
    reglas
)

### Interpretación

El código ya no depende del notebook.

Podría ser importado desde:

- otro notebook;
- un script;
- una API;
- un workflow.

# 25. Separar reglas de la lógica

In [ ]:
ruta_reglas = (
    proyecto
    / "reglas.py"
)

codigo_reglas = '''
REGLAS_DOCUMENTOS = {
    ".pdf": "PDF",
    ".xlsx": "Excel",
    ".csv": "Datos",
    ".txt": "Texto",
    ".jpg": "Imagenes",
    ".jpeg": "Imagenes",
    ".png": "Imagenes"
}
'''

ruta_reglas.write_text(
    textwrap.dedent(
        codigo_reglas
    ),
    encoding="utf-8"
)

print(
    ruta_reglas.read_text(
        encoding="utf-8"
    )
)

Separar reglas y lógica permite modificar categorías sin tocar funciones.

# 26. Crear un script de uso

In [ ]:
ruta_script = (
    proyecto
    / "main.py"
)

codigo_script = '''
from documentos import analizar_archivo
from reglas import REGLAS_DOCUMENTOS

archivos = [
    "factura.pdf",
    "datos.csv",
    "archivo.zip"
]

for archivo in archivos:
    print(
        analizar_archivo(
            archivo,
            REGLAS_DOCUMENTOS
        )
    )
'''

ruta_script.write_text(
    textwrap.dedent(
        codigo_script
    ),
    encoding="utf-8"
)

print(
    ruta_script.read_text(
        encoding="utf-8"
    )
)

# 27. Provocar un problema de responsabilidad

In [ ]:
def clasificar_y_mostrar_y_guardar(
    archivo,
    reglas_clasificacion
):
    registro = analizar_archivo(
        archivo,
        reglas_clasificacion
    )

    print(
        registro
    )

    with open(
        "resultado.txt",
        "a",
        encoding="utf-8"
    ) as fichero:
        fichero.write(
            str(
                registro
            )
            + "\n"
        )

    return registro

### Diagnóstico

La función:

- clasifica;
- imprime;
- guarda.

Tiene varias responsabilidades y será más difícil de probar.

# 28. Separar responsabilidades

In [ ]:
def formatear_registro(
    registro
):
    return (
        f'{registro["archivo"]} | '
        f'{registro["categoria"]} | '
        f'{registro["estado"]}'
    )


registro_demo = analizar_archivo(
    "factura.pdf",
    reglas
)

print(
    formatear_registro(
        registro_demo
    )
)

Ahora:

- `analizar_archivo` analiza;
- `formatear_registro` representa;
- otra función podrá guardar.

Cada cambio tiene un lugar más claro.

# 29. Función para guardar resultados

In [ ]:
def guardar_lineas(
    lineas,
    ruta_salida
):
    ruta_salida = Path(
        ruta_salida
    )

    ruta_salida.write_text(
        "\n".join(
            lineas
        )
        + "\n",
        encoding="utf-8"
    )

    return ruta_salida


lineas = [
    formatear_registro(
        registro
    )
    for registro
    in procesar_archivos(
        archivos,
        reglas
    )
]

ruta = guardar_lineas(
    lineas,
    proyecto
    / "resultado.txt"
)

print(
    ruta.read_text(
        encoding="utf-8"
    )
)

# 30. Validar una función de guardado

In [ ]:
assert ruta.exists()

contenido = ruta.read_text(
    encoding="utf-8"
)

assert (
    len(
        contenido
        .strip()
        .splitlines()
    )
    == len(
        archivos
    )
)

print(
    "Guardado validado."
)

# 31. Funciones puras y testeabilidad

Una función es especialmente sencilla de probar cuando:

- su resultado depende únicamente de sus argumentos;
- no modifica estado externo;
- no depende de variables globales;
- no realiza I/O oculto.

Ejemplo:

```python
clasificar_extension(extension, reglas)
```

es más fácil de testear que una función que clasifica, imprime, mueve y borra archivos al mismo tiempo.

# 32. Crear pruebas simples

In [ ]:
def test_obtener_extension():
    assert (
        obtener_extension(
            "FACTURA.PDF"
        )
        == ".pdf"
    )

    assert (
        obtener_extension(
            "README"
        )
        == "sin_extension"
    )


def test_clasificar_extension():
    assert (
        clasificar_extension(
            ".pdf",
            reglas
        )
        == "PDF"
    )

    assert (
        clasificar_extension(
            ".zip",
            reglas
        )
        == "Revision"
    )


test_obtener_extension()
test_clasificar_extension()

print(
    "Tests manuales superados."
)

# 33. Mini-práctica integrada

Refactoriza una automatización documental monolítica.

El resultado debe separar al menos:

```text
obtener_extension()
        ↓
clasificar_extension()
        ↓
decidir_estado()
        ↓
analizar_archivo()
        ↓
procesar_archivos()
        ↓
calcular_kpis()
        ↓
guardar_resultados()
        ↓
ejecutar_pipeline()
```

Después mueve la lógica a módulos `.py` y demuestra que puedes importar las funciones desde el notebook.

# 34. Checklist de diseño de funciones

- [ ] ¿La función tiene una responsabilidad clara?
- [ ] ¿El nombre explica qué hace?
- [ ] ¿Los parámetros son realmente necesarios?
- [ ] ¿Los valores por defecto son razonables?
- [ ] ¿Devuelve resultados reutilizables?
- [ ] ¿Evita imprimir cuando debería devolver?
- [ ] ¿Evita estado global?
- [ ] ¿Valida argumentos críticos?
- [ ] ¿Es fácil de probar?
- [ ] ¿Puede reutilizarse fuera del notebook?
- [ ] ¿La lógica está separada de configuración e I/O?

# 35. Ejercicios finales

### Ejercicio 1 — Primera función
Extrae la lógica de normalización de extensión.

### Ejercicio 2 — Parámetros
Crea una función de clasificación configurable.

### Ejercicio 3 — Valores por defecto
Añade categoría desconocida configurable.

### Ejercicio 4 — Argumentos con nombre
Ejecuta la función utilizando keyword arguments.

### Ejercicio 5 — Validación
Provoca `None` y cadena vacía y añade validaciones explícitas.

### Ejercicio 6 — Return
Crea una función que primero use `print` y después corrígela con `return`.

### Ejercicio 7 — Múltiples resultados
Devuelve total, automáticos y revisión.

### Ejercicio 8 — Diccionario de salida
Refactoriza el ejercicio anterior para devolver resultados con nombre.

### Ejercicio 9 — Globales
Construye un ejemplo con estado global y elimínalo.

### Ejercicio 10 — Composición
Combina tres funciones pequeñas.

### Ejercicio 11 — Reutilización
Usa el mismo pipeline con dos juegos de reglas diferentes.

### Ejercicio 12 — Orquestación
Crea `ejecutar_pipeline()`.

### Ejercicio 13 — Módulo
Genera `documentos.py`.

### Ejercicio 14 — Configuración
Separa las reglas en `reglas.py`.

### Ejercicio 15 — Script
Crea `main.py` que importe ambos módulos.

### Ejercicio 16 — Responsabilidad
Detecta una función que haga demasiadas cosas y divídela.

### Ejercicio 17 — Tests
Crea al menos cinco asserts para funciones pequeñas.

### Ejercicio 18 — Reto final
Construye una API interna de funciones documentales reutilizable desde notebook y script.

# 36. Preguntas de reflexión

1. ¿Qué problema resuelve una función?
2. ¿Qué significa responsabilidad única?
3. ¿Cuándo usar un parámetro por defecto?
4. ¿Qué diferencia existe entre `print` y `return`?
5. ¿Por qué los efectos secundarios dificultan las pruebas?
6. ¿Qué problema tienen las variables globales?
7. ¿Por qué devolver un diccionario puede ser mejor que una tupla?
8. ¿Qué significa componer funciones?
9. ¿Qué diferencia existe entre lógica y orquestación?
10. ¿Por qué separar reglas y código?
11. ¿Qué aporta un módulo?
12. ¿Qué ventaja tiene poder importar el código?
13. ¿Por qué las funciones pequeñas son más testeables?
14. ¿Cómo prepara este laboratorio el futuro Organizador Inteligente de Documentos?

# 37. Cierre

Ya tenemos:

```text
estructuras
    ↓
funciones
    ↓
parámetros
    ↓
retornos
    ↓
módulos
    ↓
reutilización
```

El siguiente laboratorio llevará esta lógica al sistema de archivos real:

```text
pathlib
os
shutil
glob
```

y empezaremos a crear, inspeccionar y manipular físicamente carpetas y documentos.